## Simple Neural Network


In [1]:
import pandas as pd
from main import create_hog_colour_df, scale_data
from sklearn.model_selection import train_test_split
from constants import LEAKAGE_COLS
from sklearn.preprocessing import LabelEncoder
from main import one_hot_encode

colour_hog_df = create_hog_colour_df()
X = colour_hog_df.drop(columns=LEAKAGE_COLS)
Y = colour_hog_df["class_name"]

X_train_temp, X_test, y_train_temp, y_test = train_test_split(
    X, Y, test_size=0.2, random_state=2718, stratify=Y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_temp, y_train_temp, test_size=0.2, random_state=2718, stratify=y_train_temp
)

X_train_scaled, X_val_scaled = scale_data(X_train=X_train, X_test=X_val)

label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_val_encoded = label_encoder.transform(y_val)
unique_classes = label_encoder.classes_

y_train_ohe = one_hot_encode(Y_ints=y_train_encoded, num_classes=len(unique_classes))

In [3]:
from nn_util import make_prediction, train_network, calculate_accuracy

params_dict = train_network(
    X=X_train_scaled,
    hidden_size=64,
    output_size=10,
    epochs=1000,
    Y=y_train_ohe,
    learning_rate=1,
)

predictions, probs = make_prediction(
    X=X_val_scaled, classes_array=unique_classes, params_dict=params_dict
)

df_of_preds = pd.DataFrame(predictions)
df_of_probs = pd.DataFrame(probs)
final_df = pd.concat(
    [df_of_preds, df_of_probs], axis=1
)  # adds it across row-wise, not just joined onto the bottom

# remember you have to test validation set on different learning rates and # of epochs
accuracy = calculate_accuracy(preds=predictions, true_labels=y_val)
print(f"Accuracy on validation set: {accuracy   * 100:.2f}%")


print(final_df)

Loss after 0 epochs: 2.303005326724469
Loss after 100 epochs: 0.6815681075574832
Loss after 200 epochs: 0.08384161024407223
Loss after 300 epochs: 0.031604830659671894
Loss after 400 epochs: 0.01776444312023708
Loss after 500 epochs: 0.011923876974930308
Loss after 600 epochs: 0.008805055956325207
Loss after 700 epochs: 0.006899972768564164
Loss after 800 epochs: 0.005630571287509989
Loss after 900 epochs: 0.004728863155997648
Loss after 1000 epochs: 0.004059456460016807
Accuracy on validation set: 41.50%
             0         0
0    butterfly  0.999506
1    butterfly  0.617905
2         deer  0.761910
3          dog  0.939392
4         bird  0.477900
..         ...       ...
595        dog  0.608756
596       deer  0.678096
597      horse  0.713136
598      sheep  0.848517
599   elephant  0.999957

[600 rows x 2 columns]
